In [ ]:
# DATA pro analyzu cen pozemku - vytazeni z DB VALUO
import pandas as pd
import numpy as np
import sqlalchemy

# ==========================================
# 1. PARAMETRY PRO FILTRACI
# ==========================================
datum_od = '2020-01-01'
datum_do = '2026-12-31'
okresy = ['Hlavní město Praha']

katastralni_uzemi = []
#katastralni_uzemi = ['Libeň','Kobylisy','Troja','Holešovice']

uzemni_plany = []   
#uzemni_plany = ['ZVS','ZVO','OB','OB-A','OB-B','OB-C','OB-D','OV','SV','SP','IZ']                 # Zadejte kódy/názvy funkčního využití dle vašeho číselníku (lze i více)
cena_za_m2_od = 5000         # Spodní hranice Kč/m2
cena_za_m2_do = 15000       # Horní hranice Kč/m2

params = {
    'datum_od': datum_od, 
    'datum_do': datum_do,
    'cena_od': cena_za_m2_od,
    'cena_do': cena_za_m2_do
}

# Okresy
okresy_sql = ""
if okresy:
    okresy_keys = []
    for i, okres in enumerate(okresy):
        key = f'okres_{i}'
        params[key] = okres
        okresy_keys.append(f':{key}')
    okresy_sql = f"AND V.okres IN ({', '.join(okresy_keys)})"

# Katastrální území
ku_sql = ""
if katastralni_uzemi:
    ku_keys = []
    for i, ku in enumerate(katastralni_uzemi):
        key = f'ku_{i}'
        params[key] = ku
        ku_keys.append(f':{key}')
    ku_sql = f"AND V.kat_uzemi IN ({', '.join(ku_keys)})"

# Územní plán
up_sql = ""
if uzemni_plany:
    up_keys = []
    for i, up in enumerate(uzemni_plany):
        key = f'up_{i}'
        params[key] = up
        up_keys.append(f':{key}')
    up_sql = f"AND U.POPIS_Z IN ({', '.join(up_keys)})"

# ==========================================
# 2. PŘIPOJENÍ DO MS SQL
# ==========================================
connection_string = (
    "mssql+pyodbc://LOCALHOST/VALUO"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&Trusted_Connection=yes"
)
engine = sqlalchemy.create_engine(connection_string)

# ==========================================
# 3. SQL DOTAZ S DYNAMICKÝMI FILTRY
# ==========================================
sql = f"""
WITH FilteredVklady AS (
    SELECT 
        cislo_vkladu,
        SUM(CASE WHEN nemovitost <> 'parcela' THEN 1 ELSE 0 END) as pocet_jinych_nemovitosti
    FROM Valuo_data
    GROUP BY cislo_vkladu
)
SELECT 
    CAST(V.okres AS NVARCHAR(200)) AS okres, 
    CAST(V.kat_uzemi AS NVARCHAR(200)) AS kat_uzemi, 
    CAST(K.parcel_number AS NVARCHAR(100)) AS parcelni_cislo,
    CAST(COALESCE(K.areaValue_m2, V.plocha) AS VARCHAR(100)) AS vymera, 
    CAST(U.POPIS_Z AS NVARCHAR(100)) AS uzemni_plan,
    CONVERT(VARCHAR(20), V.datum_podani, 120) AS datum_podani, 
    CAST(V.cislo_vkladu AS VARCHAR(100)) AS cislo_vkladu, 
    CAST(V.cenovy_udaj AS VARCHAR(100)) AS cenovy_udaj
FROM Valuo_data V
LEFT JOIN KN_parcel_data K ON V.id = K.id_valuo
LEFT JOIN UP_FVU_data U ON K.id_UP_FVU_data = U.id
INNER JOIN FilteredVklady FV ON V.cislo_vkladu = FV.cislo_vkladu
WHERE 
    V.nemovitost = 'parcela' 
    AND FV.pocet_jinych_nemovitosti = 0 
    AND V.datum_podani >= :datum_od AND V.datum_podani <= :datum_do
    {okresy_sql}
    {ku_sql}
    {up_sql}
    AND NULLIF(V.okres, '') IS NOT NULL
    AND NULLIF(V.kat_uzemi, '') IS NOT NULL
    AND NULLIF(K.parcel_number, '') IS NOT NULL
    AND V.datum_podani IS NOT NULL
    AND NULLIF(V.cislo_vkladu, '') IS NOT NULL
    AND V.cenovy_udaj IS NOT NULL
    AND COALESCE(K.areaValue_m2, V.plocha) IS NOT NULL
    AND (V.cenovy_udaj / NULLIF(COALESCE(K.areaValue_m2, V.plocha), 0)) >= :cena_od
    AND (V.cenovy_udaj / NULLIF(COALESCE(K.areaValue_m2, V.plocha), 0)) <= :cena_do
"""

sql_text = sqlalchemy.text(sql)

with engine.connect() as conn:
    df = pd.read_sql(sql_text, conn, params=params)

if df.empty:
    print("Žádná data neodpovídají zadaným filtrům.")
else:
    # ==========================================
    # 4. ZPRACOVÁNÍ V PANDAS
    # ==========================================
    df['vymera'] = pd.to_numeric(df['vymera'], errors='coerce')
    df['cenovy_udaj'] = pd.to_numeric(df['cenovy_udaj'], errors='coerce')
    # Výpočet jednotkové ceny pro data v Excelu (SQL nám už data omezilo, zde tvoříme jen sloupec)
    df['jednotkova_cena'] = np.where(df['vymera'] > 0, df['cenovy_udaj'] / df['vymera'], np.nan)
    
    df['LV'] = ""
    df['info_text'] = ""
    
    final_columns = [
        'okres', 'kat_uzemi', 'parcelni_cislo', 'vymera', 'uzemni_plan', 
        'LV', 'info_text', 'datum_podani', 'cislo_vkladu', 
        'cenovy_udaj', 'jednotkova_cena'
    ]
    df = df.reindex(columns=final_columns)

    # ==========================================
    # 5. OPRAVENÝ EXPORT S FYZICKÝM ZÁPISEM VZORCE
    # ==========================================
    output_path = 'select_parcel_analyza.xlsx'
    
    with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, startcol=1, sheet_name='Analyza_Pozemku')
        
        workbook = writer.book
        worksheet = writer.sheets['Analyza_Pozemku']
        
        (max_row, max_col) = df.shape
        
        formula_str = (
            '="("&CHAR(34)&[[#This Row],[okres]]&CHAR(34)&", "&CHAR(34)&[[#This Row],[kat_uzemi]]&CHAR(34)&", "&'
            'CHAR(34)&[[#This Row],[parcelni_cislo]]&CHAR(34)&", "&CHAR(34)&[[#This Row],[LV]]&CHAR(34)&", "&'
            'CHAR(34)&[[#This Row],[info_text]]&CHAR(34)&"),"'
        )
        
        column_settings = [{'header': 'python_tuple_format', 'formula': formula_str}]
        for col in df.columns:
            column_settings.append({'header': col})
            
        worksheet.add_table(0, 0, max_row, max_col, {
            'columns': column_settings,
            'name': 'DataPozemky',
            'style': 'Table Style Medium 9'
        })
        
        for row_idx in range(1, max_row + 1):
            worksheet.write_formula(row_idx, 0, formula_str)
        
        worksheet.set_column('A:A', 60)
        worksheet.set_column('B:D', 15)
        worksheet.set_column('E:G', 15)
        worksheet.set_column('H:I', 20)
        worksheet.set_column('J:L', 15)

    print(f"Hotovo. Data exportována do: {output_path}")

In [ ]:
# AUTOMATIZOVANÉ STAŽENÍ A VYKRESLENÍ PARCEL (RÚIAN / WFS INSPIRE)
import re
import json
import datetime
import urllib.parse

import requests
import pandas as pd
from lxml import etree
from pyproj import Transformer
import folium
from shapely.geometry import Polygon
from branca.element import MacroElement
from jinja2 import Template
from folium.map import Layer
from IPython.display import HTML, display
from sqlalchemy import create_engine
from openpyxl import load_workbook

# =============================================================================
# 0. PŘIPOJENÍ K DATABÁZI A POMOCNÉ FUNKCE PRO HISTORII
# =============================================================================

params_conn = urllib.parse.quote_plus(
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=localhost;"
    "Database=VALUO;"
    "Trusted_Connection=yes;"
)
connection_url = f"mssql+pyodbc:///?odbc_connect={params_conn}"
engine = create_engine(connection_url)


def get_valuo_history(okres: str, ku: str, parcelni_cislo: str, db_engine) -> dict:
    """
    Vyhledá historii parcely v DB Valuo.
    Vrací slovník s HTML výstupem pro popup, zjištěným kódem UP, max cenou,
    jednotkovou cenou (JC) a číslem vkladu odpovídajícím max. ceně.
    """
    vystup = {
        "html": "<i style='color:gray;'>Záznam v databázi Valuo nenalezen.</i>",
        "up": "Nezjištěno",
        "typ_pozemku": "Nezjištěno",
        "max_cena": None,
        "jc": None,
        "datum_podani": None,
        "cislo_vkladu": None,
    }

    db_okres = okres
    if okres.lower().strip() in ["praha", "hlavni metro praha", "praha-mesto", "praha město"]:
        db_okres = "Hlavní město Praha"

    def zformatuj_parcely(kombi_series, max_zobrazeno=5):
        items = set([str(i).strip() for i in kombi_series.dropna() if str(i).strip() and str(i).strip() != '|'])
        if not items:
            return ""

        links = []
        for item in sorted(items, key=lambda x: x.split('|')[0]):
            parts = item.split('|')
            p_num = parts[0]
            p_kod = parts[1] if len(parts) > 1 else ""

            if p_kod and p_kod.isdigit():
                url = f"https://nahlizenidokn.cuzk.gov.cz/ZobrazObjekt.aspx?&typ=parcela&id={p_kod}"
                links.append(f"<a href='{url}' target='_blank' style='color:#0066cc; text-decoration:none; font-weight:bold;' title='Otevřít v KN'>{p_num}</a>")
            else:
                links.append(p_num)

        plny_seznam = ", ".join(links)

        if len(links) > max_zobrazeno:
            nahled = ", ".join(links[:max_zobrazeno])
            zbyva = len(links) - max_zobrazeno
            return (
                f"<details style='cursor: pointer; margin-top: 2px;'>"
                f"<summary style='outline: none; color: #555;'>{nahled} ... <b style='color: #d9534f;'>(+ {zbyva} rozbalit)</b></summary>"
                f"<div style='margin-top: 4px; padding: 6px; border-left: 2px solid #d9534f; background: #f4f4f4; color: #333; line-height: 1.4; word-wrap: break-word;'>"
                f"{plny_seznam}</div>"
                f"</details>"
            )
        return plny_seznam

    # 1. Krok: Získání čísel vkladů
    query_vklady = f"""
        SELECT DISTINCT v.cislo_vkladu
        FROM Valuo_data v
        JOIN KN_parcel_data p ON v.id = p.id_valuo
        WHERE v.okres = '{db_okres}' 
          AND v.kat_uzemi = '{ku}' 
          AND p.parcel_number = '{parcelni_cislo}'
    """
    try:
        df_vklady = pd.read_sql(query_vklady, db_engine)
    except Exception as e:
        vystup["html"] = f"<div style='color:red;'>Chyba prvního dotazu: {e}</div>"
        return vystup

    if df_vklady.empty:
        return vystup

    vklady_list = tuple(df_vklady['cislo_vkladu'].tolist())
    vklady_str = f"('{vklady_list[0]}')" if len(vklady_list) == 1 else str(vklady_list)

    # 2. Krok: Dotaz pro detaily vkladu, GML_ID a ÚZEMNÍ PLÁN (UP)
    query_details = f"""
                SELECT DISTINCT 
                    v.id, 
                    p.id_UP_FVU_data,
                    v.cislo_vkladu, 
                    CONVERT(VARCHAR(10), v.datum_podani, 104) AS datum_podani, 
                    CAST(v.cenovy_udaj AS FLOAT) AS cenovy_udaj, 
                    v.nemovitost, 
                    v.typ,
                    CAST(v.plocha AS FLOAT) AS plocha,
                    CAST(p.parcel_number AS VARCHAR(100)) + '|' + ISNULL(REPLACE(CAST(p.gml_id AS VARCHAR(100)), 'CP.', ''), '') AS parcel_data_kombi,
                    u.POPIS_Z as UP
                FROM Valuo_data v
                LEFT JOIN KN_parcel_data p ON v.id = p.id_valuo
                LEFT JOIN [dbo].[UP_FVU_data] u ON p.id_UP_FVU_data = u.id
                WHERE v.cislo_vkladu IN {vklady_str}
    """

    try:
        df_details = pd.read_sql(query_details, db_engine)
    except Exception as e:
        vystup["html"] = f"<div style='color:red;'>Chyba pro stahování detailů vkladu: {e}</div>"
        return vystup

    if df_details.empty:
        return vystup

    # Extrakce kódu územního plánu (UP) a druhu pozemku (typ dle Valuo) —
    # obojí se váže ke KONKRÉTNÍ parcele (na rozdíl od ceny/JC, které se
    # váží k celé transakci/vkladu), proto se hledá řádek, jehož
    # parcel_data_kombi odpovídá dotazované parcelni_cislo.
    up_kod = "Nezjištěno"
    typ_pozemku_valuo = "Nezjištěno"
    for _, r in df_details.iterrows():
        kombi = str(r.get('parcel_data_kombi', ''))
        p_num = kombi.split('|')[0].strip()
        if p_num == str(parcelni_cislo).strip():
            if pd.notnull(r.get('UP')):
                up_kod = str(r['UP']).strip()
            if pd.notnull(r.get('typ')):
                typ_pozemku_valuo = str(r['typ']).strip()
            break
    vystup["up"] = up_kod
    vystup["typ_pozemku"] = typ_pozemku_valuo

    max_cena = df_details['cenovy_udaj'].max()
    vystup["max_cena"] = max_cena

    # 3. Krok: Zpracování HTML výstupu a extrakce JC
    html_output = ""
    jc_pro_max_cenu = None
    datum_pro_max_cenu = None
    vklad_pro_max_cenu = None

    for vklad, group in df_details.groupby('cislo_vkladu'):
        datum = str(group['datum_podani'].iloc[0]) if pd.notnull(group['datum_podani'].iloc[0]) else "Neznámé"
        cena = float(group['cenovy_udaj'].max())

        stats = group.groupby('nemovitost').agg(
            pocet=('id', 'count'),
            plocha_sum=('plocha', 'sum'),
            seznam_parcel=('parcel_data_kombi', lambda x: zformatuj_parcely(x, max_zobrazeno=5))
        ).reset_index()

        celkova_plocha = stats['plocha_sum'].sum()
        jc = cena / celkova_plocha if celkova_plocha > 0 else 0

        if cena == max_cena:
            jc_pro_max_cenu = jc
            datum_pro_max_cenu = datum
            vklad_pro_max_cenu = vklad

        html_output += f"<div style='background: #f9f9f9; border: 1px solid #ccc; padding: 5px; margin-bottom: 5px;'>"
        url_rizeni = "https://nahlizenidokn.cuzk.gov.cz/VyberRizeni.aspx"
        odkaz_vklad = f"<a href='{url_rizeni}' target='_blank' style='color:#0066cc; text-decoration:none; font-weight:bold;'>{vklad}</a>"
        html_output += f"<b>Řízení:</b> {odkaz_vklad} (ze dne {datum})<br>"
        html_output += f"<b>Kupní cena:</b> {cena:,.0f} Kč<br>".replace(',', ' ')
        html_output += f"<b>JC: {jc:,.0f} Kč/m²</b> (z plochy {celkova_plocha:,.0f} m²)<br>".replace(',', ' ')
        html_output += "<i style='font-size: 11px;'>Složení transakce:</i><br>"

        for _, r in stats.iterrows():
            html_output += f"<span style='font-size: 11px;'>- {r['nemovitost']}: {r['pocet']}x ({r['plocha_sum']:,.0f} m²)</span><br>".replace(',', ' ')
            if r['seznam_parcel']:
                html_output += f"<div style='font-size: 10px; margin-left: 12px; margin-top: 1px; margin-bottom: 4px;'>Parc. č.: {r['seznam_parcel']}</div>"
        html_output += "</div>"

    vystup["html"] = html_output
    vystup["jc"] = jc_pro_max_cenu
    vystup["datum_podani"] = datum_pro_max_cenu
    vystup["cislo_vkladu"] = vklad_pro_max_cenu
    return vystup


# =============================================================================
# 0b. MAPOVÉ SLUŽBY IPR PRAHA — ÚZEMNÍ PLÁN / METROPOLITNÍ PLÁN
# =============================================================================

MPP_TENANT = "SBTXIEUGWbqzUecw"
MPP_TILES_BASE = f"https://tiles.arcgis.com/tiles/{MPP_TENANT}/arcgis/rest/services"

MPP_SLUZBY = {
    "Z01_Vykres_zakladniho_cleneni_uzemi_NkV_2605":                              "Z01 – Výkres základního členění území",
    "Z02_Hlavni_vykres_struktury_NkV_2605":                                       "Z02 – Hlavní výkres (funkční využití území)",
    "Z03_Hlavni_vykres_infrastruktury_NkV_2605":                                  "Z03 – Hlavní výkres infrastruktury",
    "Z04_Vykres_verejne_prospesnych_staveb_opatreni_a_asanaci_NkV_2605":          "Z04 – Veřejně prospěšné stavby, opatření a asanace",
    "S01_Schema_metropolitnich_priorit_NkV_2605":                                 "S01 – Schéma metropolitních priorit",
    "S02_Schema_formalnich_rozvoju_NkV_2605":                                     "S02 – Schéma formálních rozvojů",
    "S03_Schema_vyskove_regulace_NkV_2605":                                       "S03 – Schéma výškové regulace",
    "O01_Koordinacni_vykres_NkV_2605":                                            "O01 – Koordinační výkres",
    "O02_Vykres_sirsich_vztahu_NkV_2605":                                         "O02 – Výkres širších vztahů",
    "O03_Vykres_vyhodnoceni_zaboru_ZPF_a_PUPFL_NkV_2605":                         "O03 – Vyhodnocení záboru ZPF a PUPFL",
}
MPP_VYCHOZI_ZAPNUTE = set()

MPP_MAX_NATIVE_ARCGIS_LEVEL = 13
MPP_MAX_NATIVE_ZOOM = 17

IPR_REST = "https://gs-pub.praha.eu/arcgis/rest/services"
IPR_TIMEOUT = 20
PRAHA_OBEC_KOD = 554782

VYNECHANE_STARE_VYKRESY = {
    "Výkres č. 26 - Bydelní",
    "Výkres č. 28 - Ostatní nebytové funkce",
    "Výkres č. 29 - Sport a rekreace",
    "Výkres č. 30 - Systém zeleně",
}
DATUM_STAZENI = datetime.date.today().strftime("%d.%m.%Y")

_ipr_session = requests.Session()
_ipr_cache: dict = {}


def _ipr_get(url: str, **params) -> dict:
    params.setdefault("f", "json")
    key = url + repr(sorted(params.items()))
    if key in _ipr_cache:
        return _ipr_cache[key]
    r = _ipr_session.get(url, params=params, timeout=IPR_TIMEOUT)
    r.raise_for_status()
    data = r.json()
    _ipr_cache[key] = data
    return data


def mpp_sluzba_dostupna(nazev_sluzby: str) -> bool:
    """Ověří, že daná MPP tile služba na tiles.arcgis.com skutečně odpovídá."""
    try:
        meta = _ipr_get(f"{MPP_TILES_BASE}/{nazev_sluzby}/MapServer")
        return "error" not in meta
    except Exception:
        return False


MPP_TILE_ORIGIN_X = -3.36998e7
MPP_TILE_ORIGIN_Y = 3.36998e7
MPP_TILE_SIZE = 512
MPP_RESOLUTIONS = {
    0: 198.43789687579377, 1: 132.2919312505292, 2: 92.60435187537043,
    3: 52.91677250021167, 4: 39.687579375158755, 5: 26.458386250105836,
    6: 19.843789687579378, 7: 13.229193125052918, 8: 6.614596562526459,
    9: 5.291677250021167, 10: 3.9687579375158752, 11: 2.6458386250105836,
    12: 1.9843789687579376, 13: 1.3229193125052918,
}


class _MppSJTSKTileLayer(Layer):
    """
    Vlastní dlaždicová vrstva pro MPP služby na tiles.arcgis.com.
    """

    _template = Template(u"""
        {% macro script(this, kwargs) %}
            if (typeof proj4 === 'undefined') {
                console.error('[MPP chyba] Knihovna proj4js se nenačetla — ' +
                    'zkontrolujte připojení k cdn.jsdelivr.net. Vrstvy MPP ' +
                    'nebudou fungovat.');
            } else if (!proj4.defs('EPSG:5514_MPP')) {
                proj4.defs('EPSG:5514_MPP',
                    '+proj=krovak +lat_0=49.5 +lon_0=24.8333333333333 ' +
                    '+alpha=30.2881397527778 +k=0.9999 +x_0=0 +y_0=0 ' +
                    '+ellps=bessel +towgs84=570.8,85.7,462.8,4.998,1.587,5.261,3.56 ' +
                    '+units=m +no_defs');
            }
            var {{ this.get_name() }} = new (L.TileLayer.extend({
                getTileUrl: function(coords) {
                    var R = 6378137.0;
                    var n = Math.pow(2, coords.z) * 256 / {{ this.tile_size }};
                    var leafletTileWidthM = (2 * Math.PI * R) / n;
                    var resolutions = {{ this.resolutions_js }};
                    var level = null, bestDiff = Infinity;
                    for (var lvl in resolutions) {
                        var arcgisTileWidthM = resolutions[lvl] * {{ this.tile_size }};
                        var diff = Math.abs(arcgisTileWidthM - leafletTileWidthM);
                        if (diff < bestDiff) { bestDiff = diff; level = lvl; }
                    }
                    var res = resolutions[level];
                    if (!res) { return ""; }
                    var merc_x = (coords.x + 0.5) / n * 2 * Math.PI * R - Math.PI * R;
                    var merc_y = Math.PI * R - (coords.y + 0.5) / n * 2 * Math.PI * R;
                    var lon = merc_x / R * 180 / Math.PI;
                    var lat = (2 * Math.atan(Math.exp(merc_y / R)) - Math.PI / 2) * 180 / Math.PI;
                    var sjtsk = proj4('EPSG:4326', 'EPSG:5514_MPP', [lon, lat]);
                    var sx = sjtsk[0], sy = sjtsk[1];
                    var col = Math.floor((sx - {{ this.origin_x }}) / (res * {{ this.tile_size }}));
                    var row = Math.floor(({{ this.origin_y }} - sy) / (res * {{ this.tile_size }}));
                    return "{{ this.url }}/" + level + "/" + row + "/" + col;
                }
            }))("{{ this.url }}/{z}/{y}/{x}", {
                opacity: {{ this.opacity }},
                minZoom: {{ this.min_zoom }},
                maxZoom: {{ this.max_zoom }},
                maxNativeZoom: {{ this.max_native_zoom }},
                tileSize: {{ this.tile_size }},
                attribution: "&copy; IPR Praha"
            });
            {{ this.get_name() }}.addTo({{ this._parent.get_name() }});
        {% endmacro %}
    """)

    def __init__(self, url, opacity=0.55, min_zoom=0, max_zoom=20, max_native_zoom=13,
                 resolutions=None, origin_x=MPP_TILE_ORIGIN_X, origin_y=MPP_TILE_ORIGIN_Y,
                 tile_size=MPP_TILE_SIZE):
        super().__init__()
        self._name = "MppSJTSKTileLayer"
        self.url = url
        self.opacity = opacity
        self.min_zoom = min_zoom
        self.max_zoom = max_zoom
        self.max_native_zoom = max_native_zoom
        self.origin_x = origin_x
        self.origin_y = origin_y
        self.tile_size = tile_size
        res = resolutions or MPP_RESOLUTIONS
        import json as _json
        self.resolutions_js = _json.dumps({str(k): v for k, v in res.items()})


def mpp_tile_url(nazev_sluzby: str) -> str:
    """Základ URL bez šablonových placeholderů (col/row dopočítá _MppSJTSKTileLayer)."""
    return f"{MPP_TILES_BASE}/{nazev_sluzby}/MapServer/tile"


def diagnostika_up() -> None:
    """Ověří dostupnost všech known-good MPP služeb a starého ÚPnSÚ na gs-pub.praha.eu."""
    print(f"=== MPP tile služby (tenant {MPP_TENANT}), stav k {DATUM_STAZENI} ===")
    vse_ok = True
    for nazev, popis in MPP_SLUZBY.items():
        ok = mpp_sluzba_dostupna(nazev)
        vse_ok &= ok
        print(f"  [{'OK' if ok else 'CHYBA'}] {popis:55s} {nazev}")
    if not vse_ok:
        print("  [!] Některé služby neodpovídají — IPR pravděpodobně vydal novou")
        print("      verzi s jiným sufixem než _NkV_2605. Zkuste najit_aktualni_mpp_sufix().")

    print(f"\n=== Fallback: starý ÚPnSÚ 1999 (gs-pub.praha.eu/pup) ===")
    try:
        sluzby = [s["name"].split("/")[-1] for s in _ipr_get(f"{IPR_REST}/pup").get("services", [])]
        print("  ", sluzby)
    except Exception as e:
        print(f"  [!] nedostupné: {e}")


def najit_aktualni_mpp_sufix(kandidati: list = None) -> str | None:
    """Pomocná funkce pro nalezení nového sufixu verze MPP."""
    if kandidati is None:
        import datetime as _dt
        dnes = _dt.date.today()
        kandidati = [f"_NkV_{dnes.strftime('%y%m')}", f"_NkV_{(dnes.month):02d}{dnes.year % 100:02d}"]
    for sufix in kandidati:
        nazev = f"Z02_Hlavni_vykres_struktury{sufix}"
        if mpp_sluzba_dostupna(nazev):
            return sufix
    return None


def pridej_uzemni_plany(m, jen_klicove=True, opacity=0.55, cenova_mapa=True, verbose=True,
                         pridat_mpp=True) -> str:
    """HLAVNÍ FUNKCE — přidá do mapy vrstvy ÚPD hl. m. Prahy (MPP i starý ÚPnSÚ 1999)."""
    pouzite = []

    if pridat_mpp and not getattr(m, "_mpp_proj4_pridano", False):
        m.get_root().header.add_child(folium.Element(
            '<script src="https://cdn.jsdelivr.net/npm/proj4/dist/proj4.js"></script>'
        ))
        m._mpp_proj4_pridano = True

    if pridat_mpp:
        if verbose:
            print("Ověřuji dostupnost výkresů Metropolitního plánu...")
        mpp_ok = mpp_sluzba_dostupna("Z02_Hlavni_vykres_struktury_NkV_2605")

        if not mpp_ok:
            novy_sufix = najit_aktualni_mpp_sufix()
            if novy_sufix and verbose:
                print(f"   [i] Nalezen nový sufix verze MPP: {novy_sufix} "
                      f"(původní _NkV_2605 už neplatí — upravte MPP_SLUZBY v kódu).")

        if mpp_ok:
            pridano = 0
            for nazev, popis in MPP_SLUZBY.items():
                show = nazev in MPP_VYCHOZI_ZAPNUTE
                fg = folium.FeatureGroup(name=f"MPP: {popis}", overlay=True, control=True, show=show)
                fg.add_child(_MppSJTSKTileLayer(
                    url=mpp_tile_url(nazev),
                    opacity=opacity,
                    max_zoom=20,
                    max_native_zoom=MPP_MAX_NATIVE_ZOOM,
                ))
                m.add_child(fg)
                pridano += 1
            if verbose:
                zapnute_info = (', '.join(MPP_SLUZBY[n] for n in MPP_VYCHOZI_ZAPNUTE)
                                if MPP_VYCHOZI_ZAPNUTE else "žádný (zapněte ručně v LayerControl)")
                print(f"   [+] Metropolitní plán: přidáno {pridano} výkresů (výchozí zapnuto: {zapnute_info})")
                print(f"   [i] MPP dlaždice mají nejjemnější dostupné rozlišení jen do ArcGIS "
                      f"Level {MPP_MAX_NATIVE_ARCGIS_LEVEL} (~1,3 m/px), což odpovídá Leaflet "
                      f"zoomu {MPP_MAX_NATIVE_ZOOM}. Mapa se otevírá na zoom_start=18, tedy "
                      f"jen mírné (2×) zvětšení dlaždic — pro maximální ostrost lze případně "
                      f"oddálit na zoom {MPP_MAX_NATIVE_ZOOM}.")
            pouzite.append("mpp")
        else:
            if verbose:
                print("   [!] MPP tile služby neodpovídají (žádná náhrada k dispozici).")
    elif verbose:
        print("   [i] MPP vrstvy vynechány (pridat_mpp=False) — viz mapa_parcel_MPP.html "
              "pro správně zobrazenou MPP vrstvu v nativní S-JTSK projekci.")

    stary_up_vysledek = _pridej_stary_up_fallback(m, jen_klicove=jen_klicove, opacity=opacity, verbose=verbose)
    if stary_up_vysledek:
        pouzite.append("pup")

    if cenova_mapa:
        try:
            _pridej_export_vrstvu_gspub(m, "sed", "cenova_mapa", [0, 1],
                                        "Cenová mapa stavebních pozemků HMP",
                                        opacity=0.55, show=False)
            if verbose:
                print("   [+] sed/cenova_mapa: přidána (zvýšené DPI kvůli čitelnosti popisků cen)")
        except Exception as e:
            print(f"   [!] Cenová mapa nedostupná: {e}")

    _pridej_js_diagnostiku_dlazdic(m)

    return "+".join(pouzite)


def _pridej_js_diagnostiku_dlazdic(m) -> None:
    """Naváže na KAŽDOU tile vrstvu mapy posluchač 'tileerror'/'tileload' pro diagnostiku v konzoli."""
    js = """
    <script>
    (function() {
        if (window.location.protocol === 'file:') {
            console.warn(
                '%c[MPP diagnostika] Mapa je otevřená přímo ze souboru (file://). ' +
                'Server IPR (tiles.arcgis.com) u vrstev Metropolitního plánu vrací ' +
                'CORS hlavičku vázanou na jinou doménu, a prohlížeč proto může ' +
                'dlaždice zablokovat chybou "ERR_BLOCKED_BY_ORB" — I KDYŽ SERVER ' +
                'ODPOVÍDÁ SPRÁVNĚ. Řešení: nespouštět mapu dvojklikem, ale přes ' +
                'lokální webový server, např. v terminálu ve složce s mapou ' +
                'spustit "python -m http.server 8000" a otevřít v prohlížeči ' +
                'http://localhost:8000/mapa_parcel.html.',
                'font-weight:bold;'
            );
        }
    })();
    document.addEventListener('DOMContentLoaded', function() {
        setTimeout(function() {
            for (var key in window) {
                if ((key.indexOf('tile_layer_') === 0 || key.indexOf('mpp_sjtsk_tile_layer_') === 0)
                    && window[key] && window[key].on) {
                    (function(layerVarName, layer) {
                        layer.on('tileerror', function(e) {
                            console.error('[MPP tile chyba] vrstva=' + layerVarName +
                                          ' url=' + (e.tile && e.tile.src ? e.tile.src : '?'));
                        });
                        if (key.indexOf('mpp_sjtsk_tile_layer_') === 0) {
                            layer.on('tileload', function(e) {
                                var coords = e.coords;
                                var tilePoint = layer._getTilePos ? layer._getTilePos(coords) : null;
                                console.log('[MPP tile OK] vrstva=' + layerVarName +
                                            ' leaflet(x=' + coords.x + ',y=' + coords.y + ',z=' + coords.z + ')' +
                                            ' obrazovka_px=' + (tilePoint ? ('(' + Math.round(tilePoint.x) + ',' + Math.round(tilePoint.y) + ')') : '?') +
                                            ' url=' + e.tile.src);
                            });
                        }
                    })(key, window[key]);
                }
            }
            console.log('[MPP diagnostika] Posluchače tileerror/tileload připojeny na všechny L.tileLayer vrstvy.');
        }, 500);
    });
    </script>
    """
    m.get_root().html.add_child(folium.Element(js))


class _ArcGISExportLayer(Layer):
    """Dlaždicová vrstva nad ArcGIS 'export' operací (starý ÚP i cenová mapa)."""

    _template = Template(u"""
        {% macro script(this, kwargs) %}
            var {{ this.get_name() }} = new (L.TileLayer.extend({
                getTileUrl: function(coords) {
                    var tileSize = {{ this.tile_size }};
                    var initialResolution = 2 * Math.PI * 6378137 / tileSize;
                    var originShift = 2 * Math.PI * 6378137 / 2.0;
                    var resolution = initialResolution / Math.pow(2, coords.z);
                    var minx = coords.x * tileSize * resolution - originShift;
                    var maxx = (coords.x + 1) * tileSize * resolution - originShift;
                    var miny = originShift - (coords.y + 1) * tileSize * resolution;
                    var maxy = originShift - coords.y * tileSize * resolution;
                    var bbox = [minx, miny, maxx, maxy].join(",");
                    return "{{ this.url }}?bbox=" + bbox +
                           "&bboxSR=102100&imageSR=102100&size=" + tileSize + "," + tileSize +
                           "&format=png32&transparent=true&dpi={{ this.dpi }}" +
                           "&layers={{ this.layers }}&f=image";
                }
            }))({
                opacity: {{ this.opacity }}, minZoom: {{ this.min_zoom }}, maxZoom: {{ this.max_zoom }},
                tileSize: {{ this.tile_size }},
                attribution: "&copy; IPR Praha, &copy; \u010C\u00daZK"
            });
            {{ this.get_name() }}.addTo({{ this._parent.get_name() }});
        {% endmacro %}
    """)

    def __init__(self, url, layers="show:0", opacity=0.5, min_zoom=10, max_zoom=19, tile_size=512, dpi=192):
        super().__init__()
        self._name = "ArcGISExportLayer"
        self.url, self.layers, self.opacity = url, layers, opacity
        self.min_zoom, self.max_zoom, self.tile_size, self.dpi = min_zoom, max_zoom, tile_size, dpi


def _pridej_export_vrstvu_gspub(m, slozka, sluzba, layer_ids, nazev, opacity=0.5, show=False,
                                 min_zoom=10, max_zoom=19, tile_size=512, dpi=192):
    """tile_size=512 + dpi=192 (2x retina) kvůli čitelnosti popisků."""
    url = f"{IPR_REST}/{slozka}/{sluzba}/MapServer/export"
    if isinstance(layer_ids, (list, tuple)):
        layer_ids = ",".join(str(i) for i in layer_ids)
    fg = folium.FeatureGroup(name=nazev, overlay=True, control=True, show=show)
    fg.add_child(_ArcGISExportLayer(url, layers=f"show:{layer_ids}", opacity=opacity,
                                    min_zoom=min_zoom, max_zoom=max_zoom,
                                    tile_size=tile_size, dpi=dpi))
    m.add_child(fg)
    return fg


def _pridej_stary_up_fallback(m, jen_klicove=True, opacity=0.5, verbose=True) -> str:
    """Přidá výkresy platného ÚPnSÚ hl. m. Prahy 1999 (gs-pub.praha.eu/pup)."""
    KLICOVE_VYKRESY = ["plán využití", "využití ploch", "zastaviteln", "záplav",
                       "veřejně prospěšné", "zpf", "pupfl", "hlavní výkres", "členění území"]
    try:
        sluzby = [s["name"].split("/")[-1] for s in _ipr_get(f"{IPR_REST}/pup").get("services", [])
                  if "uzemni_plan" in s["name"] or "vybrane" in s["name"]]
    except Exception as e:
        if verbose:
            print(f"   [!] Starý ÚP 1999 není dostupný: {e}")
        return ""

    if not sluzby:
        return ""

    celkem = 0
    for sluzba in sluzby:
        try:
            meta = _ipr_get(f"{IPR_REST}/pup/{sluzba}/MapServer")
        except Exception as e:
            print(f"   [!] pup/{sluzba}: {e}")
            continue
        top = [v for v in meta.get("layers", []) if v.get("parentLayerId", -1) == -1]
        vyber = [v for v in top if any(k in v["name"].lower() for k in KLICOVE_VYKRESY)] if jen_klicove else top
        if jen_klicove and not vyber:
            if verbose:
                print(f"   [i] pup/{sluzba}: žádná vrstva neodpovídá filtru KLICOVE_VYKRESY, přeskočeno "
                      f"({len(top)} vrstev k dispozici, viz jen_klicove=False pro jejich zobrazení)")
            continue
        for v in vyber:
            if v.get("name") in VYNECHANE_STARE_VYKRESY:
                if verbose:
                    print(f"   [i] pup/{sluzba}: '{v['name']}' vynechán (VYNECHANE_STARE_VYKRESY — "
                          f"hlášeno jako nefunkční, zobrazuje se jen šedá šachovnice)")
                continue
            _pridej_export_vrstvu_gspub(m, "pup", sluzba, v["id"], f"ÚP 1999: {v['name']}",
                                        opacity=opacity, show=False)
            celkem += 1
        if verbose:
            print(f"   [+] pup/{sluzba}: přidáno {celkem} z {len(top)} vrstev")
    return "pup" if celkem else ""


# =============================================================================
# 1. TŘÍDY A FUNKCE PRO VYKRESLOVÁNÍ MAPY (FOLIUM)
# =============================================================================

class BindClickRemove(MacroElement):
    def __init__(self, fg_poly_name: str, fg_text_name: str, gj_name: str, mk_name: str):
        super().__init__()
        self.fg_poly_name = fg_poly_name
        self.fg_text_name = fg_text_name
        self.gj_name = gj_name
        self.mk_name = mk_name
        self._template = Template(
            """
            {% macro script(this, kwargs) %}
            {{ this.gj_name }}.on('contextmenu', function(e) {
                {{ this.fg_poly_name }}.removeLayer({{ this.gj_name }});
                {{ this.fg_text_name }}.removeLayer({{ this.mk_name }});
            });
            {% endmacro %}
            """
        )


DISTINCT_COLORS = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#46f0f0", "#f032e6", "#bcf60c", "#fabebe",
    "#008080", "#e6beff", "#9a6324", "#fffac8", "#800000",
    "#aaffc3", "#808000", "#ffd8b1", "#000075", "#808080"
]


def _priprav_polozky_parcel(df_parcel_data: pd.DataFrame) -> tuple:
    """
    Sdílená příprava dat pro obě varianty mapy (Web Mercator i S-JTSK/MPP).
    Vrací (items, legend_list_items).
    """
    transformer = Transformer.from_crs("EPSG:5514", "EPSG:4326", always_xy=True)

    if "LV" not in df_parcel_data.columns:
        df_parcel_data["LV"] = "Neznámé"

    unique_lvs = df_parcel_data["LV"].astype(str).unique()
    lv_color_map = {lv: DISTINCT_COLORS[i % len(DISTINCT_COLORS)] for i, lv in enumerate(unique_lvs)}

    items = []
    legend_list_items = ""

    for idx, row in df_parcel_data.iterrows():
        posList_str = row.get("geometry_posList")
        if not posList_str or pd.isna(posList_str):
            continue

        coords = list(map(float, posList_str.split()))
        xy_pairs = list(zip(coords[0::2], coords[1::2]))
        lon_lat_pairs = [transformer.transform(x, y) for x, y in xy_pairs]

        poly = Polygon(lon_lat_pairs)
        if not poly.is_valid or poly.is_empty:
            continue

        centroid = (poly.centroid.y, poly.centroid.x)
        parc_label = row.get("label", "")
        area = row.get("areaValue_m2", None)
        lv = str(row.get("LV", "Neznámé"))
        okres = row.get("okres_nazev", "Neznámý")
        ku = row.get("ku_nazev", "Neznámé")
        ku_kod = row.get("ku_kod", "")
        obec_nazev = row.get("obec_nazev", "Neznámá")
        obec_kod = row.get("obec_kod", "")
        info_text = str(row.get("info", ""))
        druh_pozemku = row.get("druh_pozemku", "Nezjištěno")
        druh_pozemku_valuo = row.get("Valuo_Druh_Pozemku", "Nezjištěno")

        up_code = row.get("UP", "Nezjištěno")
        db_info_html = row.get("valuo_html", "")

        color = lv_color_map[lv]
        area_str = f"{float(area):,.0f}".replace(",", " ") if pd.notna(area) else "neznámá výměra"

        gml_id_raw = str(row.get("gml_id", ""))
        clean_id = gml_id_raw.replace("CP.", "")
        url_kn = f"https://nahlizenidokn.cuzk.gov.cz/ZobrazObjekt.aspx?&typ=parcela&id={clean_id}"

        popup_html = f"""
        <div style='font-family: sans-serif; font-size: 13px; width: 340px; line-height: 1.4;'>
            <h4 style='margin: 0 0 5px 0; border-bottom: 2px solid {color}; padding-bottom: 3px;'>
                Parcela č. <a href='{url_kn}' target='_blank' style='color:#0066cc; text-decoration:none;'>{parc_label}</a> (LV: {lv} | UP: {up_code})
            </h4>
            <b>Druh pozemku (KN):</b> {druh_pozemku}<br>
            <b>Druh pozemku (Valuo):</b> {druh_pozemku_valuo}<br>
            <b>Výměra:</b> {area_str} m²<br>
            <b>K.Ú.:</b> {ku} ({ku_kod})<br>
            <b>Obec:</b> {obec_nazev} ({obec_kod})<br>
            <b>Okres:</b> {okres}<br>
            <hr style='border: 0; border-top: 1px solid #ccc; margin: 8px 0;'>
            <h5 style='margin: 0 0 5px 0;'>Historie transakcí (Valuo DB)</h5>
            {db_info_html}
        </div>
        """
        label_html = f"LV č. {lv}<br>{parc_label}<br>{area_str} m²"

        items.append((poly, centroid, label_html, color, popup_html))

        legend_list_items += (
            f"<li style='margin-bottom: 12px; border-bottom: 1px solid #e0e0e0; padding-bottom: 6px;'>"
            f"<span style='display:inline-block; width:14px; height:14px; background-color:{color}; border:1px solid #333; margin-right:8px; vertical-align:middle;'></span>"
            f"<span style='vertical-align:middle; font-family:sans-serif;'>"
            f"LV č.{lv}, parc.č. <a href='{url_kn}' target='_blank' style='font-weight:bold; color:#0066cc; text-decoration:none;'>{parc_label}</a>, {area_str} m², okres {okres}, k.ú. {ku} [ÚP: {up_code}]"
            f"</span>"
            f"<div style='margin-left: 26px; margin-top: 4px; font-size: 11.5px; color: #444; font-style: italic; line-height: 1.4;'>{info_text}</div>"
            f"</li>"
        )

    return items, legend_list_items


def _pridej_polygony_parcel(m, items, fg_poly_name_override=None) -> tuple:
    """Vykreslí polygony + textové popisky parcel do dané mapy (sdíleno oběma variantami)."""
    fg_poly = folium.FeatureGroup(name="Polygony parcel", show=True)
    fg_text = folium.FeatureGroup(name="Popisky parcel (text)", show=True)
    fg_poly_name = fg_poly.get_name()
    fg_text_name = fg_text.get_name()

    for poly, centroid, label_html, color, popup_html in items:
        popup_okno = folium.Popup(html=popup_html, max_width=420, max_height=350, auto_close=False)
        gj = folium.GeoJson(data=poly.__geo_interface__, style_function=lambda feature, c=color: {"fillColor": c, "color": c, "weight": 2, "fillOpacity": 0.4}, tooltip="<b>Levý klik:</b> Detaily <br><b>Pravý klik:</b> Smazat polygon", popup=popup_okno).add_to(fg_poly)
        mk = folium.Marker(location=centroid, draggable=True, icon=folium.DivIcon(icon_size=(150, 54), icon_anchor=(75, 27), html=(f'<div style="font-size:10px; font-weight:bold; line-height: 1.2; text-align:center; color: black; text-shadow: 2px 2px 4px white, -1px -1px 0 white, 1px -1px 0 white, -1px 1px 0 white, 1px 1px 0 white;">{label_html}</div>'))).add_to(fg_text)
        gj.add_child(BindClickRemove(fg_poly_name=fg_poly_name, fg_text_name=fg_text_name, gj_name=gj.get_name(), mk_name=mk.get_name()))

    fg_poly.add_to(m)
    fg_text.add_to(m)
    return fg_poly, fg_text


def plot_parcels_on_map(df_parcel_data: pd.DataFrame) -> folium.Map:
    items, legend_list_items = _priprav_polozky_parcel(df_parcel_data)

    if not items:
        print("Nebyla nalezena žádná validní geometrie, vracím defaultní mapu ČR.")
        return folium.Map(location=[49.8, 15.5], zoom_start=7)

    m = folium.Map(
        location=[sum(c[0] for _, c, _, _, _ in items) / len(items),
                  sum(c[1] for _, c, _, _, _ in items) / len(items)],
        zoom_start=18, tiles=None, width="100%", height="100%", closePopupOnClick=False
    )

    folium.TileLayer(
        tiles="OpenStreetMap",
        name="Základní mapa (OSM)",
        control=True,
        show=True,
    ).add_to(m)
    folium.raster_layers.TileLayer(tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}", attr="© Google", name="Google Maps", overlay=False, control=True).add_to(m)
    folium.raster_layers.TileLayer(tiles="https://ags.cuzk.gov.cz/arcgis1/rest/services/ORTOFOTO_WM/MapServer/tile/{z}/{y}/{x}", name="ČÚZK Ortofoto", attr="© ČÚZK", overlay=False, control=True, max_zoom=20, min_zoom=6, show=False).add_to(m)
    folium.raster_layers.TileLayer(tiles="https://ags.cuzk.gov.cz/arcgis1/rest/services/ZTM_WM/MapServer/tile/{z}/{y}/{x}", attr="© ČÚZK", name="Základní topografická mapa (ČÚZK)", overlay=False, control=True, show=False).add_to(m)

    print("Připojuji mapové vrstvy IPR Praha...")
    pouzita_upd = pridej_uzemni_plany(m, jen_klicove=True, opacity=0.5, cenova_mapa=True,
                                       pridat_mpp=False)

    _pridej_polygony_parcel(m, items)
    folium.LayerControl(collapsed=False).add_to(m)

    if pouzita_upd == "mpp":
        upd_popis = ("Územní plán hl. m. Prahy (Metropolitní plán), OOP č. 001/2026, "
                     "účinný od 1. 9. 2026")
    elif pouzita_upd == "pup":
        upd_popis = "Územní plán sídelního útvaru hl. m. Prahy (1999), v platném znění"
    else:
        upd_popis = "vrstvy ÚPD nebyly připojeny"

    legend_html_container = f"""
    <div style="position: fixed; bottom: 30px; left: 30px; width: auto; max-width: 1250px; max-height: 1000px; background-color: rgba(255, 255, 255, 0.95); border: 2px solid #aaa; z-index: 9999; overflow-y: auto; padding: 10px; border-radius: 8px; box-shadow: 3px 3px 10px rgba(0,0,0,0.3);">
    <h4 style="margin-top: 0; margin-bottom: 4px; font-family: sans-serif; border-bottom: 2px solid #444; padding-bottom: 5px;">Zobrazené pozemky</h4>
    <div style="font-family: sans-serif; font-size: 10px; color: #555; margin-bottom: 8px;">
        Podklad ÚPD: {upd_popis} &nbsp;|&nbsp; datový podklad © IPR Praha, © ČÚZK &nbsp;|&nbsp; staženo {DATUM_STAZENI}
    </div>
    <ul style="list-style-type: none; padding-left: 0; margin: 0; font-size: 10px;">{legend_list_items}</ul></div>
    """
    m.get_root().html.add_child(folium.Element(legend_html_container))
    return m


def plot_parcels_on_mpp_map(df_parcel_data: pd.DataFrame) -> folium.Map:
    """
    DRUHÁ, SAMOSTATNÁ mapa — MPP (Metropolitní plán) v jeho nativní projekci
    S-JTSK/Krovák (EPSG:5514).
    """
    items, legend_list_items = _priprav_polozky_parcel(df_parcel_data)

    if not items:
        print("Nebyla nalezena žádná validní geometrie, vracím defaultní mapu ČR.")
        return folium.Map(location=[49.8, 15.5], zoom_start=7)

    m = folium.Map(
        location=[sum(c[0] for _, c, _, _, _ in items) / len(items),
                  sum(c[1] for _, c, _, _, _ in items) / len(items)],
        zoom_start=13, tiles=None, crs="EPSG3857",
        width="100%", height="100%", closePopupOnClick=False
    )

    print("Připojuji MPP vrstvy IPR Praha (nativní S-JTSK CRS)...")
    zapnute_info = []
    pridano = 0
    for nazev_sluzby, popisek in MPP_SLUZBY.items():
        if not mpp_sluzba_dostupna(nazev_sluzby):
            continue
        url = mpp_tile_url(nazev_sluzby)
        fg = folium.FeatureGroup(name=popisek, show=(nazev_sluzby in MPP_VYCHOZI_ZAPNUTE))
        folium.raster_layers.TileLayer(
            tiles=url + "/{z}/{y}/{x}", name=popisek, attr="© IPR Praha",
            overlay=True, control=True, show=(nazev_sluzby in MPP_VYCHOZI_ZAPNUTE),
            max_native_zoom=max(MPP_RESOLUTIONS.keys()), max_zoom=max(MPP_RESOLUTIONS.keys()) + 4,
            opacity=0.7, tileSize=MPP_TILE_SIZE,
        ).add_to(m)
        pridano += 1
        if nazev_sluzby in MPP_VYCHOZI_ZAPNUTE:
            zapnute_info.append(popisek)
    print(f"   [+] MPP (S-JTSK): přidáno {pridano} výkresů "
          f"(výchozí zapnuto: {', '.join(zapnute_info) if zapnute_info else 'žádný'})")

    _pridej_polygony_parcel(m, items)
    folium.LayerControl(collapsed=False).add_to(m)

    legend_html_container = f"""
    <div style="position: fixed; bottom: 30px; left: 30px; width: auto; max-width: 1250px; max-height: 1000px; background-color: rgba(255, 255, 255, 0.95); border: 2px solid #aaa; z-index: 9999; overflow-y: auto; padding: 10px; border-radius: 8px; box-shadow: 3px 3px 10px rgba(0,0,0,0.3);">
    <h4 style="margin-top: 0; margin-bottom: 4px; font-family: sans-serif; border-bottom: 2px solid #444; padding-bottom: 5px;">Zobrazené pozemky — MPP podklad (S-JTSK)</h4>
    <div style="font-family: sans-serif; font-size: 10px; color: #555; margin-bottom: 8px;">
        Podklad: Metropolitní plán hl. m. Prahy, OOP č. 001/2026, účinný od 1. 9. 2026 &nbsp;|&nbsp;
        datový podklad © IPR Praha, © ČÚZK &nbsp;|&nbsp; staženo {DATUM_STAZENI}<br>
        <b>Pozor:</b> tato mapa je v souřadnicovém systému S-JTSK a NENÍ zarovnaná
        s podkladovými mapami (Google/OSM/ČÚZK) v hlavním souboru mapy parcel.
    </div>
    <ul style="list-style-type: none; padding-left: 0; margin: 0; font-size: 10px;">{legend_list_items}</ul></div>
    """
    m.get_root().html.add_child(folium.Element(legend_html_container))

    proj4_script = (
        '<script src="https://cdn.jsdelivr.net/npm/proj4/dist/proj4.js"></script>\n'
        '<script src="https://cdn.jsdelivr.net/npm/proj4leaflet@1.0.2/src/proj4leaflet.js"></script>\n'
        '<script>\n'
        'if (typeof proj4 === "undefined" || typeof L.Proj === "undefined") {\n'
        '    console.error("[MPP S-JTSK chyba] proj4js nebo proj4leaflet se nenačetlo — '
        'zkontrolujte připojení k cdn.jsdelivr.net. Mapa nebude fungovat.");\n'
        '}\n'
        f'var MPP_SJTSK_CRS = new L.Proj.CRS("EPSG:5514",\n'
        f'    "+proj=krovak +lat_0=49.5 +lon_0=24.8333333333333 +alpha=30.2881397527778 "+\n'
        f'    "+k=0.9999 +x_0=0 +y_0=0 +ellps=bessel "+\n'
        f'    "+towgs84=570.8,85.7,462.8,4.998,1.587,5.261,3.56 +units=m +no_defs",\n'
        f'    {{\n'
        f'        origin: [{MPP_TILE_ORIGIN_X}, {MPP_TILE_ORIGIN_Y}],\n'
        f'        resolutions: {json.dumps([MPP_RESOLUTIONS[k] for k in sorted(MPP_RESOLUTIONS)])}\n'
        f'    }}\n'
        f');\n'
        '</script>\n'
    )
    m.get_root().html.add_child(folium.Element(proj4_script))

    html = m.get_root().render()
    if "crs: L.CRS.EPSG3857," not in html:
        raise RuntimeError(
            "Nepodařilo se najít očekávaný řetězec 'crs: L.CRS.EPSG3857,' ve "
            "vygenerovaném HTML — patrně se změnila verze folia/šablony. "
            "Zkontrolujte ručně inicializaci L.map() ve výstupním souboru."
        )
    html = html.replace("crs: L.CRS.EPSG3857,", "crs: MPP_SJTSK_CRS,", 1)

    class _PrerenderedMap:
        """Tenký obal, aby volající kód mohl použít stejné .save()/render() API jako folium.Map."""
        def __init__(self, rendered_html):
            self._html = rendered_html

        def save(self, path):
            with open(path, "w", encoding="utf-8") as f:
                f.write(self._html)

        def get_root(self):
            raise NotImplementedError(
                "Tato mapa je už plně vyrenderovaná (S-JTSK CRS byl vložen "
                "postprocessingem) — použijte .save(cesta) pro uložení."
            )

    return _PrerenderedMap(html)


# =============================================================================
# 2. FUNKCE PRO DOTAZOVÁNÍ RÚIAN A INSPIRE
# =============================================================================

def convert_to_gps(x: float, y: float, source_epsg: str = "EPSG:5514") -> tuple:
    transformer = Transformer.from_crs(source_epsg, "EPSG:4326", always_xy=True)
    return transformer.transform(x, y)


def get_parcel_data(okres_nazev: str, kat_uzemi_nazev: str, parcel_number: str) -> pd.DataFrame:
    PRAGUE_OBEC_KOD = 554782
    PRAGUE_VUSC_KOD = 19
    PRAGUE_ALIASES = {"praha", "hlavni metro praha", "hlavní město praha", "praha-mesto", "praha město"}

    def _norm(s: str) -> str: return (s or "").strip().lower()
    def _is_prague_okres(name: str) -> bool: return _norm(name) in PRAGUE_ALIASES
    def _sql_escape(s: str) -> str: return (s or "").replace("'", "''")

    base_url_candidates = ["https://ags.cuzk.gov.cz/arcgis/rest/services/RUIAN/Prohlizeci_sluzba_nad_daty_RUIAN/MapServer", "https://ags.cuzk.cz/ArcGIS/rest/services/RUIAN/MapServer"]
    session = requests.Session()

    def arcgis_query(base_url: str, layer_id: int, where: str, out_fields: str = "*") -> dict:
        params = {"where": where, "outFields": out_fields, "returnGeometry": "false", "f": "json"}
        r = session.get(f"{base_url}/{layer_id}/query", params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        if isinstance(data, dict) and data.get("error"): raise RuntimeError(f"ArcGIS error: {data['error'].get('message')}")
        return data

    def arcgis_query_first_ok(layer_id: int, where: str, out_fields: str = "*") -> dict:
        last_err = None
        for base_url in base_url_candidates:
            try: return arcgis_query(base_url, layer_id, where, out_fields=out_fields)
            except Exception as e: last_err = e
        raise RuntimeError(f"Chyba RÚIAN ArcGIS: {last_err}")

    praha_mode = _is_prague_okres(okres_nazev)
    okres_kod = None

    if not praha_mode:
        okres_data = arcgis_query_first_ok(layer_id=15, where=f"nazev = '{_sql_escape(okres_nazev)}'", out_fields="*")
        if not okres_data.get("features"): raise ValueError(f"Okres '{okres_nazev}' nebyl nalezen.")
        okres_kod = okres_data["features"][0]["attributes"].get("kod")

    ku_data = arcgis_query_first_ok(layer_id=7, where=f"nazev LIKE '{_sql_escape(kat_uzemi_nazev)}%'", out_fields="kod,nazev,obec")
    if not ku_data.get("features"): raise ValueError(f"K.Ú. '{kat_uzemi_nazev}' nebylo nalezeno.")

    valid_ku = []
    for f in ku_data["features"]:
        ku_atr = f["attributes"]
        obec_kod = ku_atr.get("obec")
        if obec_kod is None: continue
        obec_data = arcgis_query_first_ok(layer_id=12, where=f"kod = {int(obec_kod)}", out_fields="kod,nazev,okres")
        if not obec_data.get("features"): continue
        obec_atr = obec_data["features"][0]["attributes"]

        if praha_mode and int(obec_atr.get("kod", -1)) == PRAGUE_OBEC_KOD:
            valid_ku.append({"ku_kod": ku_atr.get("kod"), "ku_nazev": ku_atr.get("nazev"), "obec_kod": obec_atr.get("kod"), "obec_nazev": obec_atr.get("nazev")})
        elif not praha_mode and obec_atr.get("okres") == okres_kod:
            valid_ku.append({"ku_kod": ku_atr.get("kod"), "ku_nazev": ku_atr.get("nazev"), "obec_kod": obec_atr.get("kod"), "obec_nazev": obec_atr.get("nazev")})

    if not valid_ku: raise ValueError(f"Nebylo nalezeno žádné platné K.Ú.")
    selected_ku = valid_ku[0]

    if praha_mode:
        vusc_data = arcgis_query_first_ok(layer_id=17, where=f"kod = {PRAGUE_VUSC_KOD}", out_fields="kod,nazev")
        vusc_attrs = vusc_data["features"][0]["attributes"]
        okres_kod_out, okres_nazev_out = None, okres_nazev
    else:
        okres2 = arcgis_query_first_ok(layer_id=15, where=f"kod = {int(okres_kod)}", out_fields="kod,nazev,vusc")
        okres_attrs = okres2["features"][0]["attributes"]
        vusc_data = arcgis_query_first_ok(layer_id=17, where=f"kod = {int(okres_attrs.get('vusc'))}", out_fields="kod,nazev")
        vusc_attrs = vusc_data["features"][0]["attributes"]
        okres_kod_out, okres_nazev_out = okres_attrs.get("kod"), okres_attrs.get("nazev")

    params_wfs = {"service": "WFS", "version": "2.0.0", "request": "GetFeature", "storedQuery_id": "GetParcel", "UPPER_ZONING_ID": selected_ku["ku_kod"], "TEXT": parcel_number}
    resp_wfs = session.get("https://services.cuzk.cz/wfs/inspire-CP-wfs.asp", params=params_wfs, timeout=30)
    resp_wfs.raise_for_status()

    tree = etree.fromstring(resp_wfs.content)
    ns = {"wfs": "http://www.opengis.net/wfs/2.0", "gml": "http://www.opengis.net/gml/3.2", "CP": "http://inspire.ec.europa.eu/schemas/cp/4.0", "base": "http://inspire.ec.europa.eu/schemas/base/3.3"}
    parcel_elem = tree.find(".//CP:CadastralParcel", namespaces=ns)
    if parcel_elem is None: raise ValueError(f"Parcela {parcel_number} nebyla nalezena.")

    def get_text(elem, path):
        sub = elem.find(path, namespaces=ns)
        return sub.text.strip() if sub is not None and sub.text else None

    parcel_data = {
        "gml_id": parcel_elem.get("{http://www.opengis.net/gml/3.2}id"), "areaValue_m2": float(get_text(parcel_elem, "CP:areaValue") or 0),
        "beginLifespanVersion": get_text(parcel_elem, "CP:beginLifespanVersion"), "endLifespanVersion": get_text(parcel_elem, "CP:endLifespanVersion"),
        "label": get_text(parcel_elem, "CP:label"), "nationalCadastralReference": get_text(parcel_elem, "CP:nationalCadastralReference"),
        "inspire_localId": get_text(parcel_elem, "CP:inspireId/base:Identifier/base:localId"), "inspire_namespace": get_text(parcel_elem, "CP:inspireId/base:Identifier/base:namespace"),
        "refPoint_x": None, "refPoint_y": None, "refPoint_lon": None, "refPoint_lat": None,
        "geometry_posList": get_text(parcel_elem, "CP:geometry/gml:Polygon/gml:exterior/gml:LinearRing/gml:posList"),
        "ku_kod": selected_ku["ku_kod"], "ku_nazev": selected_ku["ku_nazev"], "obec_kod": selected_ku["obec_kod"], "obec_nazev": selected_ku["obec_nazev"],
        "okres_kod": okres_kod_out, "okres_nazev": okres_nazev_out, "vusc_kod": vusc_attrs.get("kod"), "vusc_nazev": vusc_attrs.get("nazev"),
    }

    ref_point = get_text(parcel_elem, "CP:referencePoint/gml:Point/gml:pos")
    if ref_point:
        coords = ref_point.split()
        if len(coords) >= 2:
            parcel_data["refPoint_x"], parcel_data["refPoint_y"] = float(coords[0]), float(coords[1])
            parcel_data["refPoint_lon"], parcel_data["refPoint_lat"] = convert_to_gps(float(coords[0]), float(coords[1]))

    druh_pozemku_text = "Nezjištěno"
    try:
        raw_id = parcel_data.get("inspire_localId", "") or parcel_data.get("gml_id", "")
        match_id = re.search(r'\d+', raw_id)
        if match_id:
            url_kn = f"https://nahlizenidokn.cuzk.gov.cz/ZobrazObjekt.aspx?&typ=parcela&id={match_id.group()}"
            res_kn = session.get(url_kn, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
            if res_kn.status_code == 200:
                match_druh = re.search(r'Druh pozemku:[\s\S]*?<td[^>]*>(.*?)</td>', res_kn.text, re.IGNORECASE)
                if match_druh: druh_pozemku_text = re.sub(r'<[^>]+>', '', match_druh.group(1)).strip()
    except Exception as e: print(f"Chyba stahování druhu pozemku: {e}")

    parcel_data["druh_pozemku"] = druh_pozemku_text
    return pd.DataFrame([parcel_data])


# =============================================================================
# 3. HLAVNÍ BLOK: ZPRACOVÁNÍ VSTUPŮ A VÝSTUP
# =============================================================================

parcely = [

("Hlavní město Praha", "Krč", "2583/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Žižkov", "3980/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Žižkov", "3982", "", "vzorek pro PM"),
("Hlavní město Praha", "Libuš", "12/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "829/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "1817/9", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "1817/15", "", "vzorek pro PM"),
("Hlavní město Praha", "Chodov", "2332/38", "", "vzorek pro PM"),
("Hlavní město Praha", "Chodov", "3503/73", "", "vzorek pro PM"),
("Hlavní město Praha", "Chodov", "8/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Chodov", "3715/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Chodov", "762", "", "vzorek pro PM"),
("Hlavní město Praha", "Chodov", "4481/9", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "3117/11", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "3117/21", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "3117/28", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "3118/9", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "3117/26", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "3100/42", "", "vzorek pro PM"),
("Hlavní město Praha", "Cholupice", "426/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Veleslavín", "299/27", "", "vzorek pro PM"),
("Hlavní město Praha", "Řepy", "1352/136", "", "vzorek pro PM"),
("Hlavní město Praha", "Řepy", "50/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "676/190", "", "vzorek pro PM"),
("Hlavní město Praha", "Chodov", "2105/117", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "2583/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "2583/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Žižkov", "4223/37", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "2566/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "2910/369", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4677", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "820/16", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "716", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4671/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4707", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4706/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4726", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4725", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4748/21", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4705/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4746/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Satalice", "404", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "2010/249", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4672/31", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4673/68", "", "vzorek pro PM"),
("Hlavní město Praha", "Letňany", "600/324", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "508", "", "vzorek pro PM"),
("Hlavní město Praha", "Žižkov", "2639/134", "", "vzorek pro PM"),
("Hlavní město Praha", "Chodov", "3027/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "3100/46", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Měcholupy", "521/407", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Měcholupy", "190", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4261/337", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "3469/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Měcholupy", "817/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Liboc", "1125/27", "", "vzorek pro PM"),
("Hlavní město Praha", "Liboc", "1125/29", "", "vzorek pro PM"),
("Hlavní město Praha", "Liboc", "1125/32", "", "vzorek pro PM"),
("Hlavní město Praha", "Liboc", "457/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Liboc", "1142/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Liboc", "1132/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "1837/155", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "1837/9", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "1293", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "5019/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4813/11", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "435", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "434", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "656", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "3009/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1004", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "2037/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Běchovice", "639/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Běchovice", "642", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "1600", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "1600", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "110/40", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "110/40", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "1142/55", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "1142/55", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "3295/24", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "3295/24", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "878/15", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "1006/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "1004/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "522/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "876/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Bubeneč", "1577/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Bubeneč", "1577/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Satalice", "261/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4813/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4813/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "3821/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Modřany", "3733/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "5047/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4706/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4705/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4748/21", "", "vzorek pro PM"),
("Hlavní město Praha", "Cholupice", "254/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Cholupice", "383/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "4612/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "2347", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "168/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "3127/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4082/26", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "493/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "637/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "1004/9", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "2973/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "2876/6", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "1004/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "607/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "2161/13", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "2704/85", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2352/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2352/20", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "1330/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "2448/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Kamýk", "254/46", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "435", "", "vzorek pro PM"),
("Hlavní město Praha", "Malešice", "793/112", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4471/21", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "42", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/103", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/89", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/123", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/102", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/124", "", "vzorek pro PM"),
("Hlavní město Praha", "Modřany", "956/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Modřany", "1717", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2527/38", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "2315/184", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "2566/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "742/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "2754/266", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "607/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4501/152", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4302/238", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4302/222", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4302/238", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4302/222", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "154/72", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "10/12", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2339/68", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2339/122", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "43/24", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2372/35", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2372/31", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "741", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2362/77", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2246", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2650/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "2552/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "2552/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "439/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4241/31", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2985/32", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2506/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2507/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2507/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2526/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "59/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Žižkov", "4439/20", "", "vzorek pro PM"),
("Hlavní město Praha", "Žižkov", "4439/23", "", "vzorek pro PM"),
("Hlavní město Praha", "Žižkov", "4439/11", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "3117/36", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "3117/30", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "3117/24", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "1656/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Letňany", "406/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "707", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "3764/13", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "432/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "3715/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "1292/149", "", "vzorek pro PM"),
("Hlavní město Praha", "Veleslavín", "299/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Veleslavín", "299/199", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "2973/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "2974/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "2973/14", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "2974/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "2899/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "2580/128", "", "vzorek pro PM"),
("Hlavní město Praha", "Veleslavín", "570/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Holešovice", "2416/85", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "2063/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "1857/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/362", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/107", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/107", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/366", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "835/38", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "835/34", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "1013/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Modřany", "1337/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2342/164", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2526/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "3715/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "762", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4481/9", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "166/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "835/89", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "835/38", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/114", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/115", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/119", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/92", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/100", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/93", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/87", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/106", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/117", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/111", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/116", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/90", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/97", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/95", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/96", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/123", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/104", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "5018/31", "", "vzorek pro PM"),
("Hlavní město Praha", "Liboc", "1759/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2887/83", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2887/83", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "172/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/69", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/72", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/68", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/70", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/71", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/78", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "349/227", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "302/103", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "196/88", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "378/114", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "349/25", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "439/52", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/73", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/74", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "5019/126", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "5019/101", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "5019/105", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "5019/104", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "5019/136", "", "vzorek pro PM"),
("Hlavní město Praha", "Žižkov", "4412/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "2283/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4547/9", "", "vzorek pro PM"),
("Hlavní město Praha", "Malešice", "872/30", "", "vzorek pro PM"),
("Hlavní město Praha", "Malešice", "950", "", "vzorek pro PM"),
("Hlavní město Praha", "Malešice", "949/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Malešice", "498/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "378/67", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "378/68", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "1365", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "907/9", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/252", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "2370/75", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2351/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2350/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "1470/176", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2350/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "817/74", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "817/73", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4323/13", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "3432/53", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3976/23", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3976/25", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3976/28", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "1653/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3976/26", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "3395/16", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "1741/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "1303/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "3093/16", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "3093/13", "", "vzorek pro PM"),
("Hlavní město Praha", "Motol", "536/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "955/61", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "1560/249", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "1560/253", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/58", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/74", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/73", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/58", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/58", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/70", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/37", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/58", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/37", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4263/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/37", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "1009/43", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "1009/68", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "2752/6", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "1349/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "2512/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/35", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4537/44", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "1079", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd nad Lesy", "4264/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1454/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1453/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "272/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "272/6", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "2641/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "2643/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "2650/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "5928/6", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "5930/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "2643/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "5930/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "3473/9", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "3148", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "3401/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Modřany", "3593/6", "", "vzorek pro PM"),
("Hlavní město Praha", "Vysočany", "1935/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Vysočany", "1935/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4241/31", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3152", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "3102/22", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2387/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "2315/290", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "5018/26", "", "vzorek pro PM"),
("Hlavní město Praha", "Letňany", "543/332", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "1861", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "59/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "5810/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Liboc", "1082/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Liboc", "1115/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4045/15", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "1739/13", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "1739/16", "", "vzorek pro PM"),
("Hlavní město Praha", "Ruzyně", "1739/29", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4408/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4085/51", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4085/45", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4085/48", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4085/42", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4085/43", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4087/21", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4087/21", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4085/43", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4085/48", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4085/51", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4420/49", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4105/9", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4036/55", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "262/44", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "1117/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "978/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "222", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "383/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "970/14", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "680/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "284", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "1866/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "13/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Modřany", "112/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Cholupice", "262/44", "", "vzorek pro PM"),
("Hlavní město Praha", "Cholupice", "1117/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Cholupice", "978/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Cholupice", "222", "", "vzorek pro PM"),
("Hlavní město Praha", "Cholupice", "383/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Cholupice", "970/14", "", "vzorek pro PM"),
("Hlavní město Praha", "Cholupice", "680/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "918/34", "", "vzorek pro PM"),
("Hlavní město Praha", "Jinonice", "877/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "676/23", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "676/24", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "287/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "657/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "657/11", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "659/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "584/12", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "658", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "659/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4380/29", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3908/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "284", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4416/26", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4085/47", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4085/46", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4087/20", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "1024/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4424/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "321", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4474/72", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4458/157", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3920/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2356/212", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2356/179", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "49/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "49/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "43/29", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2339/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "10/14", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "829/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "10/14", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2356/236", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2366/34", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "1871", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "1888/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Libuš", "318/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "1872", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2366/34", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "1866/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2368/29", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "741", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1412/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "773/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "773/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "700/228", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "1003/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "1349/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "1345/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "1347/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "680/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "1052/260", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "1154/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "894/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "378/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Vysočany", "2085", "", "vzorek pro PM"),
("Hlavní město Praha", "Vysočany", "2085", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "302/197", "", "vzorek pro PM"),
("Hlavní město Praha", "Žižkov", "2931/288", "", "vzorek pro PM"),
("Hlavní město Praha", "Žižkov", "2931/289", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2986/54", "", "vzorek pro PM"),
("Hlavní město Praha", "Háje", "536/347", "", "vzorek pro PM"),
("Hlavní město Praha", "Háje", "519/102", "", "vzorek pro PM"),
("Hlavní město Praha", "Háje", "576/228", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "127", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "127", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "2763/9", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "160/630", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "1168/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "2041/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "157/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "241/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/122", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/112", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/98", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/118", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/110", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/75", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/94", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/101", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/99", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/88", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4672/42", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "3471", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "879/16", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4679/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Vršovice", "1342", "", "vzorek pro PM"),
("Hlavní město Praha", "Malešice", "1025/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Malešice", "1141", "", "vzorek pro PM"),
("Hlavní město Praha", "Malešice", "493/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Malešice", "637/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Malešice", "749/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/71", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/72", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/68", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/70", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/69", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2575/26", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2575/346", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/69", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/72", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/68", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/70", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2671/71", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2575/153", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "185/6", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "378/75", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "378/76", "", "vzorek pro PM"),
("Hlavní město Praha", "Kyje", "2886/163", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "242/17", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "1653/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "283/30", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "252/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "1656/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "3117/36", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "3117/26", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "2825/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Braník", "1913/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/78", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "155/232", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "155/234", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "2131/574", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "1937", "", "vzorek pro PM"),
("Hlavní město Praha", "Letňany", "524/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Letňany", "526/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "382/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "439/238", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "1292/84", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "2654/56", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "2654/58", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "10/11", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "43/22", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "43/22", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "44/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "49/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "43/28", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "243", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2349/15", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2352/12", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "132/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "49/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "43/27", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2356/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "1015/53", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "507/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "669/12", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "669/15", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2368/30", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2372/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "49/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "43/26", "", "vzorek pro PM"),
("Hlavní město Praha", "Stodůlky", "67", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/105", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/107", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/120", "", "vzorek pro PM"),
("Hlavní město Praha", "Točná", "399/91", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "3640/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "2747/71", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "2747/73", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "2286", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "2747/71", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "654/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "1619/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "654/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "4781/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "4246/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "3071/18", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "1230/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4468/56", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3976/26", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3976/23", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3976/25", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "729", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "703/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "737", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4416/18", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4423/56", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "1967/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3976/23", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3976/25", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3976/22", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "3976/26", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4244/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4394/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "214/66", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "214/66", "", "vzorek pro PM"),
("Hlavní město Praha", "Šeberov", "1470/176", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "214/67", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "214/503", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "2574/26", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "1676/123", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "2404/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "504/13", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "1767/47", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "1003/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "1349/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "1345/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "1347/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "680/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Dejvice", "1052/260", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "3009/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "185", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "2869/379", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "3314/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "925/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "3139/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1833/14", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "894/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2114/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Modřany", "1652/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Nové Město", "2537/162", "", "vzorek pro PM"),
("Hlavní město Praha", "Nové Město", "1908/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Šeberov", "1465/23", "", "vzorek pro PM"),
("Hlavní město Praha", "Šeberov", "263/24", "", "vzorek pro PM"),
("Hlavní město Praha", "Šeberov", "263/23", "", "vzorek pro PM"),
("Hlavní město Praha", "Šeberov", "1407/39", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "214/556", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4669", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4669", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4670/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4705/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4671/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4669", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4706/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4705/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4670/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4669", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4676", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4706/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "49/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "43/31", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "43/22", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "741", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "2230/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "2921", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "639/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/113", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/358", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/107", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/364", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/359", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/107", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/365", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/107", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/363", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/112", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/368", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/107", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/367", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/107", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/107", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/360", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/107", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/108", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "845/361", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "3129/6", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1307/55", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1307/126", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4471/20", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "2585/35", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "2450/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2986/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "4086/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2986/13", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "570", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "570", "", "vzorek pro PM"),
("Hlavní město Praha", "Chodov", "2105/113", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "2303/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Holešovice", "2410/37", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "3295/22", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "3295/22", "", "vzorek pro PM"),
("Hlavní město Praha", "Střešovice", "781/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Břevnov", "2532/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Břevnov", "138/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "2235/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Troja", "1336/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Troja", "1336/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4748/21", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4706/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4705/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4707", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4746/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4725", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4726", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4671/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Horní Počernice", "4241/182", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "2634/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2986/50", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4705/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4748/21", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4676", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4725", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4706/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4726", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4707", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4671/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4014/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2443", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2225/333", "", "vzorek pro PM"),
("Hlavní město Praha", "Háje", "536/536", "", "vzorek pro PM"),
("Hlavní město Praha", "Nové Město", "2313/20", "", "vzorek pro PM"),
("Hlavní město Praha", "Nové Město", "2313/15", "", "vzorek pro PM"),
("Hlavní město Praha", "Nové Město", "2313/16", "", "vzorek pro PM"),
("Hlavní město Praha", "Nové Město", "2313/17", "", "vzorek pro PM"),
("Hlavní město Praha", "Nové Město", "2313/18", "", "vzorek pro PM"),
("Hlavní město Praha", "Nové Město", "2313/19", "", "vzorek pro PM"),
("Hlavní město Praha", "Nové Město", "2313/21", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "920/62", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "1198", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "1362/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Holešovice", "1000/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Holešovice", "996/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "340/13", "", "vzorek pro PM"),
("Hlavní město Praha", "Troja", "1403/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Troja", "1403/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Troja", "1403/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4549/12", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "680/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "439/52", "", "vzorek pro PM"),
("Hlavní město Praha", "Štěrboholy", "348/6", "", "vzorek pro PM"),
("Hlavní město Praha", "Háje", "584/15", "", "vzorek pro PM"),
("Hlavní město Praha", "Háje", "584/12", "", "vzorek pro PM"),
("Hlavní město Praha", "Staré Město", "878/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "1427/62", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "1560/253", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "1560/249", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "1475/332", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "842", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "3098/31", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2985/27", "", "vzorek pro PM"),
("Hlavní město Praha", "Troja", "441", "", "vzorek pro PM"),
("Hlavní město Praha", "Troja", "441", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1307/58", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1307/132", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1307/40", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1307/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1307/53", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1307/56", "", "vzorek pro PM"),
("Hlavní město Praha", "Michle", "1307/99", "", "vzorek pro PM"),
("Hlavní město Praha", "Modřany", "4175/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Modřany", "4175/31", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "3334/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "3334/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2641/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2643/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2650/4", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "5928/6", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "5930/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2643/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "5930/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "2000/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "2000/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2986/49", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2986/59", "", "vzorek pro PM"),
("Hlavní město Praha", "Háje", "246/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Újezd u Průhonic", "214/555", "", "vzorek pro PM"),
("Hlavní město Praha", "Šeberov", "718/9", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "1544/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "1198", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2985/28", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2988/15", "", "vzorek pro PM"),
("Hlavní město Praha", "Troja", "758/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4671/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4669", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4670/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4707", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4671/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4706/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4705/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4748/21", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4669", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4670/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4706/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4670/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4671/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4705/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4669", "", "vzorek pro PM"),
("Hlavní město Praha", "Střížkov", "48/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Žižkov", "4436", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "16", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "16", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "2893/60", "", "vzorek pro PM"),
("Hlavní město Praha", "Krč", "2893/60", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4943/2", "", "vzorek pro PM"),
("Hlavní město Praha", "Holešovice", "1000/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Holešovice", "1059/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "1547/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2986/56", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2986/54", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/49", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/46", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/48", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/50", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/54", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/55", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/58", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/59", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/69", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/62", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/66", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/67", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/63", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/68", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "647/51", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "3700/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "2910/365", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "127", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "127", "", "vzorek pro PM"),
("Hlavní město Praha", "Hloubětín", "1651/182", "", "vzorek pro PM"),
("Hlavní město Praha", "Strašnice", "4549/12", "", "vzorek pro PM"),
("Hlavní město Praha", "Zličín", "675/25", "", "vzorek pro PM"),
("Hlavní město Praha", "Zličín", "675/25", "", "vzorek pro PM"),
("Hlavní město Praha", "Zličín", "675/51", "", "vzorek pro PM"),
("Hlavní město Praha", "Zličín", "675/51", "", "vzorek pro PM"),
("Hlavní město Praha", "Košíře", "328/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Troja", "232/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Troja", "231/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "4/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Písnice", "992/6", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2362/77", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "2910/253", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4606/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4707", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "4671/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Záběhlice", "2650/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Modřany", "265/49", "", "vzorek pro PM"),
("Hlavní město Praha", "Nové Město", "1908/1", "", "vzorek pro PM"),
("Hlavní město Praha", "Nové Město", "1908/3", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "2110/8", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "1433/10", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "1433/11", "", "vzorek pro PM"),
("Hlavní město Praha", "Hostivař", "551", "", "vzorek pro PM"),
("Hlavní město Praha", "Háje", "450/7", "", "vzorek pro PM"),
("Hlavní město Praha", "Kunratice", "1110/5", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "2838/32", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "3203", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "2958/6", "", "vzorek pro PM"),
("Hlavní město Praha", "Nusle", "3192", "", "vzorek pro PM"),
("Hlavní město Praha", "Smíchov", "820/16", "", "vzorek pro PM"),
("Hlavní město Praha", "Libeň", "2408", "", "vzorek pro PM"),



]

dfs = []
print("Načítám data z API ČÚZK...")

for okres, ku, parc, lv, info in parcely:
    try:
        df_one = get_parcel_data(okres, ku, parc)
        df_one["LV"] = str(lv); df_one["info"] = str(info)
        dfs.append(df_one)
    except ValueError as ve: print(f"⚠️ Upozornění: {ve} (k.ú. {ku}) -> Přeskakuji."); continue
    except Exception as e: print(f"❌ Chyba API pro parcelu {parc}: {e} -> Přeskakuji."); continue

if not dfs:
    print("❌ Kritická chyba: Nepodařilo se stáhnout data pro žádnou zadanou parcelu.")
else:
    df_parcel_data = pd.concat(dfs, ignore_index=True)

    print("Dotazuji DB Valuo pro získání cen, jednotkových cen (JC), čísel vkladů, druhu pozemku a kódů územního plánu (UP)...")
    valuo_html_list, up_list, max_cena_list, jc_list, datum_podani_list = [], [], [], [], []
    cislo_vkladu_list = []
    typ_pozemku_valuo_list = []

    for idx, row in df_parcel_data.iterrows():
        vysledek_valuo = get_valuo_history(row.get("okres_nazev", "Neznámý"), row.get("ku_nazev", "Neznámé"), row.get("label", ""), engine)

        valuo_html_list.append(vysledek_valuo["html"])
        up_list.append(vysledek_valuo["up"])
        max_cena_list.append(vysledek_valuo["max_cena"])
        jc_list.append(vysledek_valuo["jc"])
        datum_podani_list.append(vysledek_valuo["datum_podani"])
        cislo_vkladu_list.append(vysledek_valuo["cislo_vkladu"])
        typ_pozemku_valuo_list.append(vysledek_valuo["typ_pozemku"])

    # Zápis všech dat přímo do hlavního DataFrame
    df_parcel_data["valuo_html"] = valuo_html_list
    df_parcel_data["UP"] = up_list
    df_parcel_data["Valuo_Cislo_Vkladu"] = cislo_vkladu_list
    df_parcel_data["Valuo_Druh_Pozemku"] = typ_pozemku_valuo_list
    df_parcel_data["Valuo_Max_Cena_Kč"] = max_cena_list
    df_parcel_data["Valuo_JC_Kč_m2"] = jc_list
    df_parcel_data["Valuo_Datum_Podani"] = datum_podani_list
    df_parcel_data["datum_stazeni"] = DATUM_STAZENI       # přezkoumatelnost dle § 52 vyhl. 503/2020 Sb.

    # url_kn = odkaz na Nahlížení do KN
    df_parcel_data["url_kn"] = df_parcel_data["gml_id"].fillna("").astype(str).str.replace("CP.", "", regex=False).apply(
        lambda cid: f"https://nahlizenidokn.cuzk.gov.cz/ZobrazObjekt.aspx?&typ=parcela&id={cid}" if cid else ""
    )

    # --- Příprava dat pro Excel ---------------------------------------------
    df_export = (
        df_parcel_data
        .assign(
            parcelni_cislo=lambda d: d["label"],
            lat=lambda d: d["refPoint_lat"],
            lon=lambda d: d["refPoint_lon"],
        )
        .loc[:, ["parcelni_cislo", "Valuo_Druh_Pozemku", "areaValue_m2", "UP",
                 "okres_nazev", "ku_nazev", "Valuo_Cislo_Vkladu",
                 "LV",
                 "Valuo_Max_Cena_Kč", "Valuo_JC_Kč_m2", "Valuo_Datum_Podani", "geometry_posList",
                 "info", "lat", "lon", "url_kn", "datum_stazeni"]]
        .rename(columns={
            "parcelni_cislo": "Parc.č.",
            "Valuo_Druh_Pozemku": "Druh pozemku",
            "areaValue_m2": "Výměra [m²]",
            "UP": "ÚP",
            "okres_nazev": "Okres",
            "ku_nazev": "Katastralní území",
            "Valuo_Cislo_Vkladu": "Číslo vkladu",
            "LV": "LV č.",
            "Valuo_Max_Cena_Kč": "Max. cena Valuo [Kč]",
            "Valuo_JC_Kč_m2": "JC Valuo [Kč/m²]",
            "Valuo_Datum_Podani": "Datum podání (Valuo)",
            "geometry_posList": "Geometrie",
            "info": "Informace",
            "lat": "Zeměpisná šířka",
            "lon": "Zeměpisná délka",
            "url_kn": "URL Nahlížení do KN",
            "datum_stazeni": "Datum stažení",
        })
    )

    df_export["Zeměpisná šířka"] = df_export["Zeměpisná šířka"].round(8)
    df_export["Zeměpisná délka"] = df_export["Zeměpisná délka"].round(8)

    # Uložení DataFramu do Excelu
    out_path = "data_parcel.xlsx"
    df_export.to_excel(out_path, index=False, sheet_name="parcely_gps")

    # --- VLOŽENÍ HYPERLINKU DO SLOUPCE "Parc.č." ---
    wb = load_workbook(out_path)
    ws = wb["parcely_gps"]

    header = [cell.value for cell in ws[1]]
    col_parc = header.index("Parc.č.") + 1
    col_url = header.index("URL Nahlížení do KN") + 1

    for row in range(2, ws.max_row + 1):
        cell = ws.cell(row=row, column=col_parc)
        url_value = ws.cell(row=row, column=col_url).value
        if url_value:
            cell.hyperlink = url_value
            cell.style = "Hyperlink"

    wb.save(out_path)
    print(f"Data uložena do: {out_path}")

    print("Generuji mapu...")
    m = plot_parcels_on_map(df_parcel_data)

    html_file = "mapa_parcel.html"
    m.save(html_file)
    print(f"Interaktivní mapa uložena do: {html_file}")
    display(HTML(m._repr_html_()))

    print("Generuji MPP mapu (samostatný soubor, S-JTSK CRS)...")
    m_mpp = plot_parcels_on_mpp_map(df_parcel_data)
    html_file_mpp = "mapa_parcel_MPP.html"
    m_mpp.save(html_file_mpp)
    print(f"MPP mapa (S-JTSK) uložena do: {html_file_mpp}")

Načítám data z API ČÚZK...
⚠️ Upozornění: Parcela 3715/2 nebyla nalezena. (k.ú. Chodov) -> Přeskakuji.
⚠️ Upozornění: Parcela 4481/9 nebyla nalezena. (k.ú. Chodov) -> Přeskakuji.
⚠️ Upozornění: Parcela 1125/27 nebyla nalezena. (k.ú. Liboc) -> Přeskakuji.
⚠️ Upozornění: Parcela 1125/29 nebyla nalezena. (k.ú. Liboc) -> Přeskakuji.
⚠️ Upozornění: Parcela 3821/1 nebyla nalezena. (k.ú. Smíchov) -> Přeskakuji.
⚠️ Upozornění: Parcela 493/10 nebyla nalezena. (k.ú. Strašnice) -> Přeskakuji.
⚠️ Upozornění: Parcela 835/89 nebyla nalezena. (k.ú. Písnice) -> Přeskakuji.
⚠️ Upozornění: Parcela 2370/75 nebyla nalezena. (k.ú. Písnice) -> Přeskakuji.
⚠️ Upozornění: Parcela 1470/176 nebyla nalezena. (k.ú. Kunratice) -> Přeskakuji.
⚠️ Upozornění: Parcela 2643/1 nebyla nalezena. (k.ú. Michle) -> Přeskakuji.
⚠️ Upozornění: Parcela 2650/4 nebyla nalezena. (k.ú. Michle) -> Přeskakuji.
⚠️ Upozornění: Parcela 5928/6 nebyla nalezena. (k.ú. Michle) -> Přeskakuji.
⚠️ Upozornění: Parcela 5930/5 nebyla nalezena. (k